In [1]:
import pathlib
from collections import OrderedDict
from typing import Any, Iterable

import ray
import numpy as np
import anndata
import pandas as pd
# from bolero.tl.dataset.sc_transforms import FilterRegions
# from bolero.tl.dataset.transforms import (
#     FetchRegionOneHot,
#     ReverseComplement,
# )

# from bolero.utils import get_global_coords, understand_regions
# from bolero.tl.dataset.file_transforms import FetchRegionBigWigs, GetEmbedding
# from utils import BorzoiRegions, clamp_sqrt_large_value

# DNA_NAME = "dna_one_hot"
from bolero.tl.model.borzoi.dataset import BorzoiDatasetOnline
from bolero import init

init(num_cpus=32, object_store_memory_gb=100)

2024-10-11 11:09:36,507	INFO worker.py:1762 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [2]:
data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
bw_file_path = f'{data_dir}bigwig/'
cell_types = ['ASC', 'Vip']
# bed = f'{data_dir}peaks/cCREs.bed'


In [3]:
bigwig_paths = [f'{bw_file_path}'+f'{ct}.bw' for ct in cell_types]
bigwig_paths

['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw',
 '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw']

In [4]:
bigwig_names_dict = OrderedDict()
for path in bigwig_paths:
    bigwig_names_dict[pathlib.Path(path).name.rsplit('.',1)[0]] = path


assert len(bigwig_names_dict) == len(bigwig_paths)

In [5]:
for i, x in enumerate(bigwig_names_dict):
    print(x)
    print(bigwig_names_dict[x])

ASC
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw
Vip
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw


In [6]:
bigwig_paths_dict = OrderedDict()

In [7]:
for path in bigwig_paths:
    cell_type = path.split('/')[-1]
    cell_type = cell_type.rsplit('.',1)[0]
    bigwig_paths_dict[cell_type] = path

In [8]:
path = "/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw"

# Remove the leading and trailing quotes
# path = path.strip("'")

# Split the path and get the last part
filename_with_extension = path.split('/')[-1]

# Remove the extension
filename_without_extension = filename_with_extension.rsplit('.', 1)[0]

filename_without_extension

'ASC'

In [9]:
adata = anndata.read_h5ad('/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad')
scvi_embedding = adata.obsm['X_scVI']
# Create a DataFrame with embeddings and cell types
df = pd.DataFrame(scvi_embedding, index=adata.obs.index)
df['cell_type'] = adata.obs['MajorType']
# Group by cell type
grouped = df.groupby('cell_type').mean()
# Get embedding
recalculated_embedding = grouped.to_numpy()


leg_map = {item: index for index, item in enumerate(grouped.index.to_list())}
leg_map

/tmp/ipykernel_1563137/135511446.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


{'ASC': 0,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

In [10]:
for x in cell_types: print(recalculated_embedding[leg_map[x]])

[ 8.1639951e-01 -1.8141804e-02  1.9753073e-01 -7.8901976e-02
 -4.7131974e-02  3.6194551e-01  4.6953613e-01 -5.1410848e-01
  3.9528120e-01  3.6116695e-01  3.5134324e-01 -3.3298790e-01
  8.2313013e-01 -1.3897213e-04 -2.4195077e-01 -1.2614359e+00
 -1.4791901e-02  8.7709546e-01 -5.9044462e-01  5.2708077e-01
  1.1471124e+00  7.0392508e-03  8.5427901e-03  1.2751074e+00
 -9.8622002e-02 -1.2545434e-02 -1.3628805e-01  3.1145185e-01
  8.4211981e-01 -3.4372029e-01  2.4858487e-01 -2.3774694e-01
 -1.6763769e+00 -3.4681487e-01 -4.7062245e-01  6.3714623e-01
  2.5193891e-01  1.8790249e+00  1.5217294e+00  3.6817715e-01
 -1.1824822e+00  3.0063364e-01 -3.6512426e-01  4.6925202e-01
 -6.6868114e-01  2.3944591e-01  9.7631979e-01 -4.0132409e-01
  4.8463538e-01  6.4016777e-01]
[-6.93263650e-01  2.43766513e-02  5.12231112e-01 -1.10374182e-01
  3.21036965e-01  4.99607325e-02 -5.97075462e-01  5.64686716e-01
 -1.59261763e-01 -2.85740286e-01  2.75409430e-01  6.66004956e-01
  2.10795522e-01  1.94720123e-02 -6.71250

In [11]:
# borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=bed, leg_map=leg_map, genome="hg38")

In [12]:
# borzoi_online_dataset.train()

In [13]:
# borzoi_online_dataset.get_default_config()

In [14]:
# vars(borzoi_online_dataset)

In [15]:
# loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=bed)

# for batch in loader1:
#     pass

# for k, v in batch.items():
#     print(k, v.dtype, v.shape)
# del loader1

In [16]:
leg_map['ASC'] = 1841
leg_map

{'ASC': 1841,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

## PROCESSED DATA ##

In [17]:
borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=None, leg_map=leg_map, genome="hg38", use_borzoi_regions=True)
borzoi_online_dataset.train()

In [18]:
borzoi_online_dataset.get_default_config()

{'bigwig_paths': 'REQUIRED',
 'bed': 'REQUIRED',
 'leg_map': 'REQUIRED',
 'lora': 'REQUIRED',
 'genome': 'hg38',
 'batch_size': 2,
 'dna_window': 524288,
 'pos_resolution': 32,
 'reverse_complement': True,
 'max_jitter': 0,
 'clamp_sqrt_threshold': None,
 'shuffle_files': True,
 'read_parquet_kwargs': None,
 'borzoi_regions': False}

In [19]:
borzoi_online_dataset.lora

bool

In [20]:
vars(borzoi_online_dataset)

{'genome': Genome: hg38
 Fasta Path: /home/tlgallent/projects/software/bolero/src/bolero/data/hg38/fasta/hg38.fa
 Genome One Hot Zarr: Not created,
 'batch_size': 2,
 'dna': False,
 '_block_size': 20,
 '_max_blocks': 200,
 'bigwig_paths': ['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw',
  '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw'],
 'bigwig_names_dict': OrderedDict([('ASC',
               '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw'),
              ('Vip',
               '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw')]),
 'keys': list[str],
 'leg_map': {'ASC': 1841,
  'Amy': 1,
  'CHD7': 2,
  'EC': 3,
  'Foxp2': 4,
  'L4_IT': 5,
  'L5_ET': 6,
  'L5_IT': 7,
  'L6_CT': 8,
  'L6_IT': 9,
  'L6_IT_Car3': 10,
  'L6b': 11,
  'L23_IT': 12,
  'L56_NP': 13,
  'Lamp5': 14,
  'Lamp5_LHX6': 15,
  'MGC': 16,
  'MSN_D1': 17,
  'MSN_D2': 18,
  'ODC': 19,
  'OPC': 20,
  'PC': 21,
  'Pvalb': 22,
  'Pvalb_ChC': 23,

In [21]:
borzoi_online_dataset.dataset_mode

'train'

In [22]:
(train_folds, valid_folds, test_folds, train_regions, valid_regions, test_regions) = borzoi_online_dataset.get_train_valid_test(fold=0)

In [23]:
train_regions

,Chromosome,Start,End,Name,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [24]:
loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=train_regions)

In [25]:
borzoi_online_dataset.bed

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [26]:
for batch in loader1:
    pass

print('printing batch items...')
for k, v in batch.items():
    print(k, v.dtype, v.shape)
del loader1

/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2024-10-11 11:09:54,700	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-11_11-09-33_628057_1563137/logs/ray-data
2024-10-11 11:09:54,701	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> AllToAllOperator[Repartition]


Loading genome DNA one-hot encoding...
Created remote one-hot object at ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000002e1f505)
Data loader kwargs {'batch_size': 2, 'prefetch_batches': 3, 'drop_last': True}


2024-10-11 11:10:32,295	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-11_11-09-33_628057_1563137/logs/ray-data
2024-10-11 11:10:32,296	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(select_columns)->MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(_concat_bw_chunks)->MapBatches(FetchRegionOneHot)] -> TaskPoolMapOperator[MapBatches(ReverseComplement)->MapBatches(_region_to_global_coords)] -> TaskPoolMapOperator[FlatMap(_convert_data)]


printing batch items...
cell_type_id torch.int64 torch.Size([2])
bw_values torch.float32 torch.Size([2, 2, 16384])
dna_one_hot torch.bool torch.Size([2, 4, 524288])
Original_Name torch.int64 torch.Size([2])
region torch.int64 torch.Size([2, 2])


In [27]:
batch['cell_type_id']

tensor([  28, 1841])

## CHECK REGIONS AND DNA ONE HOT MANUAL ##

In [28]:
list(reversed(sorted(list(batch['bw_values'][1,0,:]))))[:15]

[tensor(2.4062),
 tensor(2.1875),
 tensor(2.1250),
 tensor(1.8750),
 tensor(1.0938),
 tensor(0.9688),
 tensor(0.9375),
 tensor(0.9062),
 tensor(0.8438),
 tensor(0.7188),
 tensor(0.6250),
 tensor(0.6250),
 tensor(0.5938),
 tensor(0.5625),
 tensor(0.5625)]

In [29]:
batch['bw_values'][0,:,:6]

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0312, 0.0625],
        [0.0000, 0.0000, 0.0000, 0.0312, 0.0312, 0.0000]])

In [30]:
batch['dna_one_hot'][1,:,:322]

tensor([[False,  True, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [ True, False, False,  ..., False,  True,  True],
        [False, False,  True,  ...,  True, False, False]])

In [31]:
batch['Original_Name'][0]

tensor(52596)

In [32]:
batch['Original_Name'][1]

tensor(52883)

In [33]:
batch['region'][1]

tensor([2911336118, 2911860406])

In [34]:
batch['region'][0]

tensor([2911434464, 2911958752])

In [35]:
from bolero import Genome

In [36]:
genome = Genome('hg38')

In [37]:
genome.parse_global_coords(batch['region'].cpu().numpy())

,Chromosome,Start,End,Name
0,chrX,36432942,36957230,chrX:36432942-36957230
1,chrX,36334596,36858884,chrX:36334596-36858884


In [38]:
train_regions[train_regions['Original_Name'] == 52596]

,Chromosome,Start,End,region,Original_Name,fold
39793,chrX,36432942,36957230,chrX:36432942-36957230,52596,7


In [46]:
train_regions[train_regions['Original_Name'] == 52883]

,Chromosome,Start,End,region,Original_Name,fold
39794,chrX,36334596,36858884,chrX:36334596-36858884,52883,7


In [39]:
borzoi_online_dataset.bigwig_names_dict

OrderedDict([('ASC',
              '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw'),
             ('Vip',
              '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw')])

In [47]:
atac_bw_path = borzoi_online_dataset.bigwig_names_dict['ASC']
chrom = "chrX"
start = "36334596"
end = "36858884"

In [48]:
import pyBigWig
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, int(start), int(end), numpy=True))

data[:129]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [49]:
batch['bw_values'][1,0,:4]

tensor([0.0000, 0.0625, 0.0312, 0.0312])

In [50]:
sum(data[:32])/32

0.0

In [51]:
sum(data[32:64])/32

0.0625

In [52]:
sum(data[64:96])/32

0.03125

In [53]:
sum(data[96:128])/32

0.03125

In [43]:
atac_bw_path = borzoi_online_dataset.bigwig_names_dict['Vip']
chrom = "chrX"
start = "36432942"
end = "36957230"

In [44]:
import pyBigWig
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, int(start), int(end), numpy=True))

data[:129]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0.], dtype=float32)

In [45]:
batch['bw_values'][1,1,:4]

tensor([0.0000, 0.0000, 0.0312, 0.0000])

## CHECK REVERSE COMPLEMENT MANUAL ##